# *Severity-sweep robustness results + rotated_memory_x circuit spot-check*


In [ ]:
!pip install stim pymatching json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [ ]:

import numpy as np
import pymc as pm
import arviz as az
from scipy import stats as sstats
from scipy.special import logsumexp, softmax
import math
import warnings
warnings.filterwarnings('ignore')

RNG_SEED = 12345
np.random.seed(RNG_SEED)

SAMPLE_CORES = 1

print("Setup complete.")


E_IDEAL = 1.0 / np.sqrt(2)
COEFF_NET = E_IDEAL

A_TRUE = E_IDEAL
B_TRUE = -0.053
C_TRUE = 0.45

B1_TRUE, C1_TRUE = -0.06, 0.50
B2_TRUE, C2_TRUE = -0.015, 0.12

DISTANCES = np.array([5, 7, 9, 11, 13], dtype=float)

def true_PL_single(d):
    E = A_TRUE + B_TRUE * np.exp(-C_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def true_PL_double(d):
    E = A_TRUE + B1_TRUE * np.exp(-C1_TRUE * d) + B2_TRUE * np.exp(-C2_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def evaluate_tstate_properties(PL_x, n_shots):
    E_T = COEFF_NET * (1.0 - 2.0 * PL_x)
    var_pm = (COEFF_NET**2) * (4.0 * PL_x * (1.0 - PL_x) / n_shots)
    var_x0 = 0.5 / n_shots
    return E_T, np.sqrt(var_pm + var_x0)

def generate_synthetic_data(distances, n_shots, truth_type='single', rng=None):
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    E_obs, sigma_obs = [], []
    for d in distances:
        pl_true = true_PL_single(d) if truth_type == 'single' else true_PL_double(d)
        n_errors = rng.binomial(n_shots, pl_true)
        pl_obs = n_errors / n_shots
        e_t, sig = evaluate_tstate_properties(pl_obs, n_shots)
        E_obs.append(e_t)
        sigma_obs.append(sig)
    return np.array(E_obs), np.array(sigma_obs)

print("Ground truths loaded.")


def build_single_exp_model(d_obs, E_obs, sigma_obs):
    with pm.Model() as model:
        A = pm.Uniform("A", lower=0.65, upper=0.75)
        log_neg_B = pm.Normal("log_neg_B", mu=-1.0, sigma=1.5)
        B = pm.Deterministic("B", -pm.math.exp(log_neg_B))
        log_C = pm.Normal("log_C", mu=0.0, sigma=1.0)
        C = pm.Deterministic("C", pm.math.exp(log_C))
        E_pred = A + B * pm.math.exp(-C * d_obs)
        pm.Normal("obs", mu=E_pred, sigma=sigma_obs, observed=E_obs)
    return model

def build_double_exp_model(d_obs, E_obs, sigma_obs):
    with pm.Model() as model:
        A = pm.Uniform("A", lower=0.65, upper=0.75)
        log_neg_B1 = pm.Normal("log_neg_B1", mu=-1.5, sigma=1.0)
        B1 = pm.Deterministic("B1", -pm.math.exp(log_neg_B1))
        log_C1 = pm.Normal("log_C1", mu=0.3, sigma=0.7)
        C1 = pm.Deterministic("C1", pm.math.exp(log_C1))
        log_neg_B2 = pm.Normal("log_neg_B2", mu=-3.0, sigma=1.0)
        B2 = pm.Deterministic("B2", -pm.math.exp(log_neg_B2))
        frac = pm.Beta("frac_C2_of_C1", alpha=2, beta=5)
        C2 = pm.Deterministic("C2", C1 * frac)
        E_pred = A + B1 * pm.math.exp(-C1 * d_obs) + B2 * pm.math.exp(-C2 * d_obs)
        pm.Normal("obs", mu=E_pred, sigma=sigma_obs, observed=E_obs)
    return model

def fit_with_diagnostics(model, draws, tune, target_accept=0.95):
    """Runs NUTS and returns (trace, diagnostics_dict) instead of just a
    trace, so bad fits are visible instead of silently trusted."""
    with model:
        trace = pm.sample(draws=draws, tune=tune, chains=2,
                           target_accept=target_accept, cores=SAMPLE_CORES,
                           random_seed=RNG_SEED, progressbar=False)
    var_names = [v for v in ["A", "B", "C", "B1", "C1", "B2", "C2"]
                 if v in trace.posterior]
    summary = az.summary(trace, var_names=var_names)
    diag = {
        "divergences": int(trace.sample_stats.diverging.sum()),
        "min_ess_bulk": float(summary["ess_bulk"].min()),
        "max_rhat": float(summary["r_hat"].max()),
    }
    diag["clean"] = (diag["divergences"] == 0 and diag["min_ess_bulk"] >= 400
                      and diag["max_rhat"] <= 1.01)
    return trace, diag

print("Model builders ready.")


def prior_only_baseline(model_builder, d_obs, prob=0.95, draws=4000):
    """Samples A from the prior alone (no likelihood) and checks whether
    the resulting interval already contains A_TRUE."""
    dummy_E = np.zeros(len(d_obs))
    dummy_sigma = np.ones(len(d_obs))
    model = model_builder(d_obs, dummy_E, dummy_sigma)
    with model:
        prior = pm.sample_prior_predictive(draws=draws, random_seed=RNG_SEED)
    A_prior = prior.prior["A"].values.flatten()

    hdi = az.hdi(A_prior, prob=prob)
    contains = bool(hdi[0] <= A_TRUE <= hdi[1])
    return {"hdi_low": float(hdi[0]), "hdi_high": float(hdi[1]), "contains_A_TRUE": contains}

baseline_single = prior_only_baseline(build_single_exp_model, DISTANCES)
print("Prior-only baseline (single-exp model):", baseline_single)


def run_coverage_test(n_trials, n_shots, truth_type, draws, tune, target_accept=0.95):
    rng = np.random.default_rng(RNG_SEED + 9999)
    results, diagnostics = [], []

    for i in range(n_trials):
        E_obs, sigma_obs = generate_synthetic_data(DISTANCES, n_shots, truth_type, rng)

        model = build_single_exp_model(DISTANCES, E_obs, sigma_obs)

        trace, diag = fit_with_diagnostics(model, draws=draws, tune=tune,
                                            target_accept=target_accept)
        A_samples = trace.posterior["A"].values.flatten()
        hdi = az.hdi(A_samples, prob=0.95)
        contains = bool(hdi[0] <= A_TRUE <= hdi[1])

        results.append(contains)
        diagnostics.append(diag)

        if (i + 1) % 10 == 0:
            n_dirty = sum(not d["clean"] for d in diagnostics)
            print(f"  Trial {i+1}/{n_trials} done. ({n_dirty} trials flagged unclean so far)")

    coverage = np.mean(results)
    n_success = int(sum(results))
    ci_low, ci_high = sstats.beta.ppf([0.025, 0.975], n_success + 1, n_trials - n_success + 1)
    n_dirty = sum(not d["clean"] for d in diagnostics)

    print(f"\n--- Coverage Test ({truth_type} truth, {n_shots} shots) ---")
    print(f"Coverage: {coverage*100:.1f}% ({n_success}/{n_trials})")
    print(f"95% Bayesian credible interval on coverage: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
    print(f"Trials flagged with sampling problems (divergences / low ESS / high rhat): {n_dirty}/{n_trials}")
    if n_dirty > 0:
        print("Coverage recomputed excluding flagged trials:",
              f"{np.mean([r for r, d in zip(results, diagnostics) if d['clean']])*100:.1f}%")

    return results, diagnostics

res_correct, diag_correct = run_coverage_test(
    n_trials=150, n_shots=1_000_000, truth_type='single', draws=2000, tune=1500)
res_misspec, diag_misspec = run_coverage_test(
    n_trials=150, n_shots=1_000_000, truth_type='double', draws=2000, tune=1500)


rng_check = np.random.default_rng(RNG_SEED + 555)
E_chk, sigma_chk = generate_synthetic_data(DISTANCES, 1_000_000, 'double', rng_check)
model_chk = build_double_exp_model(DISTANCES, E_chk, sigma_chk)
trace_chk, diag_chk = fit_with_diagnostics(model_chk, draws=500, tune=800, target_accept=0.97)
print("Double-exp model, correctly specified, geometry check:", diag_chk)


import stim
import pymatching

def logical_error_rate(distance, rounds, p, shots, seed):
    """STAND-IN circuit. Replace with your actual injection/measurement
    circuit before trusting this cell's output."""
    circuit = stim.Circuit.generated(
        'surface_code:rotated_memory_z',
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler(seed=seed)
    det, obs = sampler.sample(shots=shots, separate_observables=True)
    pred = matcher.decode_batch(det)
    n_errors = int(np.sum(pred[:, 0] != obs[:, 0]))
    return n_errors / shots

def validate_variance_formula(distance, p, n_shots, repeats, rounds=None):
    """Runs the circuit `repeats` independent times at fixed distance,
    computes the empirical variance of E_T across those repeats, and
    compares it to the theoretical formula evaluated at the mean PL."""
    if rounds is None:
        rounds = int(distance)
    E_vals = []
    for r in range(repeats):
        pl = logical_error_rate(distance, rounds, p, n_shots, seed=RNG_SEED + r)
        e_t, _ = evaluate_tstate_properties(pl, n_shots)
        E_vals.append(e_t)
    E_vals = np.array(E_vals)
    empirical_var = np.var(E_vals, ddof=1)
    mean_pl = np.mean([(1 - e / COEFF_NET) / 2 for e in E_vals])
    _, theo_sigma = evaluate_tstate_properties(mean_pl, n_shots)
    theo_var = theo_sigma ** 2
    return {
        "distance": distance, "mean_PL": mean_pl,
        "empirical_var": empirical_var, "theoretical_var": theo_var,
        "ratio": empirical_var / theo_var,
    }

for d in [5, 9, 13]:
    result = validate_variance_formula(distance=d, p=0.005, n_shots=5000, repeats=30)
    print(result)


def compute_exact_loo(d_arr, E_obs, sigma_obs, model_builder, model_name):
    n = len(d_arr)
    log_preds = []
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        d_train, E_train, sig_train = d_arr[mask], E_obs[mask], sigma_obs[mask]

        model = model_builder(d_train, E_train, sig_train)
        trace, diag = fit_with_diagnostics(model, draws=2500, tune=2000, target_accept=0.95)
        if not diag["clean"]:
            print(f"  WARNING: {model_name} fold {i+1}/{n} did not sample cleanly: {diag}")

        A_s = trace.posterior["A"].values.flatten()
        if "B" in trace.posterior:
            B_s = trace.posterior["B"].values.flatten()
            C_s = trace.posterior["C"].values.flatten()
            E_pred_s = A_s + B_s * np.exp(-C_s * d_arr[i])
        else:
            B1_s = trace.posterior["B1"].values.flatten()
            C1_s = trace.posterior["C1"].values.flatten()
            B2_s = trace.posterior["B2"].values.flatten()
            C2_s = trace.posterior["C2"].values.flatten()
            E_pred_s = A_s + B1_s * np.exp(-C1_s * d_arr[i]) + B2_s * np.exp(-C2_s * d_arr[i])

        log_liks = sstats.norm.logpdf(E_obs[i], loc=E_pred_s, scale=sigma_obs[i])
        log_pred = logsumexp(log_liks) - np.log(len(log_liks))
        log_preds.append(log_pred)
        print(f"  {model_name} Fold {i+1}/{n}: log_pred={log_pred:.4f}")

    return float(np.sum(log_preds))

def compare_models(d_arr, E_obs, sigma_obs):
    loo_single = compute_exact_loo(d_arr, E_obs, sigma_obs, build_single_exp_model, "Single")
    loo_double = compute_exact_loo(d_arr, E_obs, sigma_obs, build_double_exp_model, "Double")

    # Primary result: raw exact LOO, no extra penalty.
    weights_raw = softmax([loo_single, loo_double])
    print("\n--- Model Comparison (PRIMARY: raw exact LOO) ---")
    print(f"Raw LOO Single: {loo_single:.4f}, Raw LOO Double: {loo_double:.4f}")
    print(f"Model Weights: Single={weights_raw[0]:.3f}, Double={weights_raw[1]:.3f}")

    n = len(d_arr)
    penalty_single, penalty_double = (3/2)*math.log(n), (5/2)*math.log(n)
    weights_pen = softmax([loo_single - penalty_single, loo_double - penalty_double])
    print("\n--- Comparison (REJECTED: BIC-penalty stacked on exact LOO) ---")
    print(f"Penalized weights: Single={weights_pen[0]:.3f}, Double={weights_pen[1]:.3f}")
    print("Rejected because: exact LOO already penalizes overfitting via held-out")
    print("prediction; adding a second (k/2)log(n) penalty double-counts complexity")
    print("cost, and here the penalty gap (%.3f) exceeds the raw predictive gap (%.3f)."
          % (penalty_double - penalty_single, loo_double - loo_single))

    return {"loo_single": loo_single, "loo_double": loo_double,
            "weights_raw": weights_raw.tolist(), "weights_penalized": weights_pen.tolist()}

rng_loo = np.random.default_rng(RNG_SEED + 777)
E_obs_loo, sigma_obs_loo = generate_synthetic_data(DISTANCES, 1_000_000, 'double', rng_loo)
comparison_result = compare_models(DISTANCES, E_obs_loo, sigma_obs_loo)

def true_PL_double_scaled(d, severity):
    """Same as true_PL_double, but B2's amplitude is scaled by `severity`.
    severity=1 reproduces your original misspecification exactly."""
    B2 = B2_TRUE * severity
    E = A_TRUE + B1_TRUE * np.exp(-C1_TRUE * d) + B2 * np.exp(-C2_TRUE * d)
    return np.clip((1.0 - E / COEFF_NET) / 2.0, 1e-6, 0.49)

def generate_synthetic_data_scaled(distances, n_shots, severity, rng):
    E_obs, sigma_obs = [], []
    for d in distances:
        pl_true = true_PL_double_scaled(d, severity)
        n_errors = rng.binomial(n_shots, pl_true)
        pl_obs = n_errors / n_shots
        e_t, sig = evaluate_tstate_properties(pl_obs, n_shots)
        E_obs.append(e_t)
        sigma_obs.append(sig)
    return np.array(E_obs), np.array(sigma_obs)

def severity_sweep_coverage(severity, n_trials, n_shots, draws=2000, tune=1500,
                             target_accept=0.98):
    """Same as run_coverage_test, but also tracks bias (posterior mean - truth)
    and interval width per trial, not just the coverage boolean."""
    rng = np.random.default_rng(RNG_SEED + 9999)
    biases, widths, covers, diagnostics = [], [], [], []
    n_failed = 0
    for i in range(n_trials):
        try:
            E_obs, sigma_obs = generate_synthetic_data_scaled(DISTANCES, n_shots, severity, rng)
            model = build_single_exp_model(DISTANCES, E_obs, sigma_obs)
            trace, diag = fit_with_diagnostics(model, draws=draws, tune=tune,
                                                target_accept=target_accept)
            A_s = trace.posterior["A"].values.flatten()
            hdi = az.hdi(A_s, prob=0.95)
            biases.append(float(np.mean(A_s)) - A_TRUE)
            widths.append(float(hdi[1] - hdi[0]))
            covers.append(bool(hdi[0] <= A_TRUE <= hdi[1]))
            diagnostics.append(diag)
        except Exception as e:

            n_failed += 1
            print(f"  [severity={severity}] trial {i+1}/{n_trials} FAILED: "
                  f"{type(e).__name__}: {e}")
            continue

        if (i + 1) % 10 == 0:
            print(f"  [severity={severity}] {i+1}/{n_trials} done "
                  f"({n_failed} failed so far)")

    n_ok = len(covers)
    n_dirty = sum(not d["clean"] for d in diagnostics)
    result = {
        "severity": severity,
        "coverage": float(np.mean(covers)) if n_ok else None,
        "mean_bias": float(np.mean(biases)) if n_ok else None,
        "bias_std": float(np.std(biases)) if n_ok else None,
        "mean_interval_width": float(np.mean(widths)) if n_ok else None,
        "n_flagged": n_dirty,
        "n_failed": n_failed,
        "n_trials": n_trials,
    }
    print(result)
    return result


sweep_results = []
for severity in [1, 3, 6, 10, 20, 40, 80]:
    r = severity_sweep_coverage(severity, n_trials=30, n_shots=1_000_000)
    sweep_results.append(r)



Setup complete.
Ground truths loaded.
Model builders ready.
Prior-only baseline (single-exp model): {'hdi_low': 0.6562481537912734, 'hdi_high': 0.7497392830036317, 'contains_A_TRUE': True}
  Trial 10/150 done. (0 trials flagged unclean so far)
  Trial 20/150 done. (0 trials flagged unclean so far)


Shape validation failed: input_shape: (1, 2000), minimum_shape: (chains=2, draws=4)


  Trial 30/150 done. (2 trials flagged unclean so far)
  Trial 40/150 done. (2 trials flagged unclean so far)


ERROR:pymc.stats.convergence:There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


  Trial 50/150 done. (3 trials flagged unclean so far)


...

ERROR:pymc.stats.convergence:There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


  Double Fold 4/5: log_pred=6.1286


ERROR:pymc.stats.convergence:There were 7 divergences after tuning. Increase `target_accept` or reparameterize.


  Double Fold 5/5: log_pred=5.7643

--- Model Comparison (PRIMARY: raw exact LOO) ---
Raw LOO Single: 27.8141, Raw LOO Double: 28.9536
Model Weights: Single=0.242, Double=0.758

--- Comparison (REJECTED: BIC-penalty stacked on exact LOO) ---
Penalized weights: Single=0.615, Double=0.385
Rejected because: exact LOO already penalizes overfitting via held-out
prediction; adding a second (k/2)log(n) penalty double-counts complexity
cost, and here the penalty gap (1.609) exceeds the raw predictive gap (1.140).
  [severity=1] 10/30 done (0 failed so far)
  [severity=1] 20/30 done (0 failed so far)
  [severity=1] 30/30 done (0 failed so far)
{'severity': 1, 'coverage': 0.5666666666666667, 'mean_bias': -0.0025096186584890544, 'bias_std': 0.00014601245471085623, 'mean_interval_width': 0.004796061753049833, 'n_flagged': 0, 'n_failed': 0, 'n_trials': 30}


NameError: name 'json' is not defined

In [ ]:
sweep_results = []
for severity in [1, 3, 6, 10, 20, 40, 80]:
    r = severity_sweep_coverage(severity, n_trials=30, n_shots=1_000_000)
    sweep_results.append(r)

  [severity=1] 10/30 done (0 failed so far)
  [severity=1] 20/30 done (0 failed so far)
  [severity=1] 30/30 done (0 failed so far)
{'severity': 1, 'coverage': 0.5666666666666667, 'mean_bias': -0.0025096186584890544, 'bias_std': 0.00014601245471085623, 'mean_interval_width': 0.004796061753049833, 'n_flagged': 0, 'n_failed': 0, 'n_trials': 30}
  [severity=3] 10/30 done (0 failed so far)
  [severity=3] 20/30 done (0 failed so far)
  [severity=3] 30/30 done (0 failed so far)
{'severity': 3, 'coverage': 0.7666666666666667, 'mean_bias': -0.004452232237137573, 'bias_std': 0.0005523470527504215, 'mean_interval_width': 0.010091733462283436, 'n_flagged': 3, 'n_failed': 0, 'n_trials': 30}
  [severity=6] 10/30 done (0 failed so far)
  [severity=6] 20/30 done (0 failed so far)
  [severity=6] 30/30 done (0 failed so far)
{'severity': 6, 'coverage': 0.8666666666666667, 'mean_bias': -0.0059625853668236325, 'bias_std': 0.001040370811761848, 'mean_interval_width': 0.014582360105635886, 'n_flagged': 0, 

In [ ]:
import stim
p = 0.005 # error probability
distance = 5
circuit_d5 = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=3 * distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p / 10,
    before_measure_flip_probability=0.05 * p,
    after_reset_flip_probability=0.2 * p
)
print(f"Circuit for distance = {distance}:")
print(circuit_d5)

Circuit for distance = 5:
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(1, 3) 12
QUBIT_COORDS(2, 2) 13
QUBIT_COORDS(3, 3) 14
QUBIT_COORDS(4, 2) 15
QUBIT_COORDS(5, 3) 16
QUBIT_COORDS(6, 2) 17
QUBIT_COORDS(7, 3) 18
QUBIT_COORDS(8, 2) 19
QUBIT_COORDS(9, 3) 20
QUBIT_COORDS(10, 2) 21
QUBIT_COORDS(0, 4) 22
QUBIT_COORDS(1, 5) 23
QUBIT_COORDS(2, 4) 24
QUBIT_COORDS(3, 5) 25
QUBIT_COORDS(4, 4) 26
QUBIT_COORDS(5, 5) 27
QUBIT_COORDS(6, 4) 28
QUBIT_COORDS(7, 5) 29
QUBIT_COORDS(8, 4) 30
QUBIT_COORDS(9, 5) 31
QUBIT_COORDS(1, 7) 34
QUBIT_COORDS(2, 6) 35
QUBIT_COORDS(3, 7) 36
QUBIT_COORDS(4, 6) 37
QUBIT_COORDS(5, 7) 38
QUBIT_COORDS(6, 6) 39
QUBIT_COORDS(7, 7) 40
QUBIT_COORDS(8, 6) 41
QUBIT_COORDS(9, 7) 42
QUBIT_COORDS(10, 6) 43
QUBIT_COORDS(0, 8) 44
QUBIT_COORDS(1, 9) 45
QUBIT_COORDS(2, 8) 46
QUBIT_COORDS(3, 9) 47
QUBIT_COORDS(4, 8) 48
QUBIT_COORDS(5, 9) 49
QUBIT_COORDS(6, 8) 50
QUBIT_COORD

In [ ]:
import stim
p = 0.005 # error probability
distance = 7
circuit_d7 = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=3 * distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p / 10,
    before_measure_flip_probability=0.05 * p,
    after_reset_flip_probability=0.2 * p
)
print(f"Circuit for distance = {distance}:")
print(circuit_d7)

Circuit for distance = 7:
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(10, 0) 10
QUBIT_COORDS(11, 1) 11
QUBIT_COORDS(13, 1) 13
QUBIT_COORDS(1, 3) 16
QUBIT_COORDS(2, 2) 17
QUBIT_COORDS(3, 3) 18
QUBIT_COORDS(4, 2) 19
QUBIT_COORDS(5, 3) 20
QUBIT_COORDS(6, 2) 21
QUBIT_COORDS(7, 3) 22
QUBIT_COORDS(8, 2) 23
QUBIT_COORDS(9, 3) 24
QUBIT_COORDS(10, 2) 25
QUBIT_COORDS(11, 3) 26
QUBIT_COORDS(12, 2) 27
QUBIT_COORDS(13, 3) 28
QUBIT_COORDS(14, 2) 29
QUBIT_COORDS(0, 4) 30
QUBIT_COORDS(1, 5) 31
QUBIT_COORDS(2, 4) 32
QUBIT_COORDS(3, 5) 33
QUBIT_COORDS(4, 4) 34
QUBIT_COORDS(5, 5) 35
QUBIT_COORDS(6, 4) 36
QUBIT_COORDS(7, 5) 37
QUBIT_COORDS(8, 4) 38
QUBIT_COORDS(9, 5) 39
QUBIT_COORDS(10, 4) 40
QUBIT_COORDS(11, 5) 41
QUBIT_COORDS(12, 4) 42
QUBIT_COORDS(13, 5) 43
QUBIT_COORDS(1, 7) 46
QUBIT_COORDS(2, 6) 47
QUBIT_COORDS(3, 7) 48
QUBIT_COORDS(4, 6) 49
QUBIT_COORDS(5, 7) 50
QUBIT_COORDS(6, 6) 51
Q

In [ ]:
import stim
p = 0.005 # error probability
distance = 9
circuit_d9 = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=3 * distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p / 10,
    before_measure_flip_probability=0.05 * p,
    after_reset_flip_probability=0.2 * p
)
print(f"Circuit for distance = {distance}:")
print(circuit_d9)

Circuit for distance = 9:
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(10, 0) 10
QUBIT_COORDS(11, 1) 11
QUBIT_COORDS(13, 1) 13
QUBIT_COORDS(14, 0) 14
QUBIT_COORDS(15, 1) 15
QUBIT_COORDS(17, 1) 17
QUBIT_COORDS(1, 3) 20
QUBIT_COORDS(2, 2) 21
QUBIT_COORDS(3, 3) 22
QUBIT_COORDS(4, 2) 23
QUBIT_COORDS(5, 3) 24
QUBIT_COORDS(6, 2) 25
QUBIT_COORDS(7, 3) 26
QUBIT_COORDS(8, 2) 27
QUBIT_COORDS(9, 3) 28
QUBIT_COORDS(10, 2) 29
QUBIT_COORDS(11, 3) 30
QUBIT_COORDS(12, 2) 31
QUBIT_COORDS(13, 3) 32
QUBIT_COORDS(14, 2) 33
QUBIT_COORDS(15, 3) 34
QUBIT_COORDS(16, 2) 35
QUBIT_COORDS(17, 3) 36
QUBIT_COORDS(18, 2) 37
QUBIT_COORDS(0, 4) 38
QUBIT_COORDS(1, 5) 39
QUBIT_COORDS(2, 4) 40
QUBIT_COORDS(3, 5) 41
QUBIT_COORDS(4, 4) 42
QUBIT_COORDS(5, 5) 43
QUBIT_COORDS(6, 4) 44
QUBIT_COORDS(7, 5) 45
QUBIT_COORDS(8, 4) 46
QUBIT_COORDS(9, 5) 47
QUBIT_COORDS(10, 4) 48
QUBIT_COORDS(11, 5) 49
QUBIT_COORDS(12, 4

In [ ]:
import stim
p = 0.005 # error probability
distance = 11
circuit_d11 = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=3 * distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p / 10,
    before_measure_flip_probability=0.05 * p,
    after_reset_flip_probability=0.2 * p
)
print(f"Circuit for distance = {distance}:")
print(circuit_d11)

Circuit for distance = 11:
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(10, 0) 10
QUBIT_COORDS(11, 1) 11
QUBIT_COORDS(13, 1) 13
QUBIT_COORDS(14, 0) 14
QUBIT_COORDS(15, 1) 15
QUBIT_COORDS(17, 1) 17
QUBIT_COORDS(18, 0) 18
QUBIT_COORDS(19, 1) 19
QUBIT_COORDS(21, 1) 21
QUBIT_COORDS(1, 3) 24
QUBIT_COORDS(2, 2) 25
QUBIT_COORDS(3, 3) 26
QUBIT_COORDS(4, 2) 27
QUBIT_COORDS(5, 3) 28
QUBIT_COORDS(6, 2) 29
QUBIT_COORDS(7, 3) 30
QUBIT_COORDS(8, 2) 31
QUBIT_COORDS(9, 3) 32
QUBIT_COORDS(10, 2) 33
QUBIT_COORDS(11, 3) 34
QUBIT_COORDS(12, 2) 35
QUBIT_COORDS(13, 3) 36
QUBIT_COORDS(14, 2) 37
QUBIT_COORDS(15, 3) 38
QUBIT_COORDS(16, 2) 39
QUBIT_COORDS(17, 3) 40
QUBIT_COORDS(18, 2) 41
QUBIT_COORDS(19, 3) 42
QUBIT_COORDS(20, 2) 43
QUBIT_COORDS(21, 3) 44
QUBIT_COORDS(22, 2) 45
QUBIT_COORDS(0, 4) 46
QUBIT_COORDS(1, 5) 47
QUBIT_COORDS(2, 4) 48
QUBIT_COORDS(3, 5) 49
QUBIT_COORDS(4, 4) 50
QUBIT_COORDS

In [ ]:
import stim
p = 0.005 # error probability
distance = 13
circuit_d13 = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=3 * distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p / 10,
    before_measure_flip_probability=0.05 * p,
    after_reset_flip_probability=0.2 * p
)
print(f"Circuit for distance = {distance}:")
print(circuit_d13)

Circuit for distance = 13:
QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(6, 0) 6
QUBIT_COORDS(7, 1) 7
QUBIT_COORDS(9, 1) 9
QUBIT_COORDS(10, 0) 10
QUBIT_COORDS(11, 1) 11
QUBIT_COORDS(13, 1) 13
QUBIT_COORDS(14, 0) 14
QUBIT_COORDS(15, 1) 15
QUBIT_COORDS(17, 1) 17
QUBIT_COORDS(18, 0) 18
QUBIT_COORDS(19, 1) 19
QUBIT_COORDS(21, 1) 21
QUBIT_COORDS(22, 0) 22
QUBIT_COORDS(23, 1) 23
QUBIT_COORDS(25, 1) 25
QUBIT_COORDS(1, 3) 28
QUBIT_COORDS(2, 2) 29
QUBIT_COORDS(3, 3) 30
QUBIT_COORDS(4, 2) 31
QUBIT_COORDS(5, 3) 32
QUBIT_COORDS(6, 2) 33
QUBIT_COORDS(7, 3) 34
QUBIT_COORDS(8, 2) 35
QUBIT_COORDS(9, 3) 36
QUBIT_COORDS(10, 2) 37
QUBIT_COORDS(11, 3) 38
QUBIT_COORDS(12, 2) 39
QUBIT_COORDS(13, 3) 40
QUBIT_COORDS(14, 2) 41
QUBIT_COORDS(15, 3) 42
QUBIT_COORDS(16, 2) 43
QUBIT_COORDS(17, 3) 44
QUBIT_COORDS(18, 2) 45
QUBIT_COORDS(19, 3) 46
QUBIT_COORDS(20, 2) 47
QUBIT_COORDS(21, 3) 48
QUBIT_COORDS(22, 2) 49
QUBIT_COORDS(23, 3) 50
QUBIT_COORDS(24, 2) 51
QUBIT_C

In [ ]:
import stim
import pymatching
import numpy as np

RNG_SEED = 12345
E_IDEAL = 1.0 / np.sqrt(2)

def run_memory_plus(distance, rounds, p, shots, seed):
    """Run |+⟩ memory experiment, return PL (logical error rate for X)."""
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=p,
        before_round_data_depolarization=p / 10,
        before_measure_flip_probability=0.05 * p,
        after_reset_flip_probability=0.2 * p,
    )
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler(seed=seed)
    det, obs = sampler.sample(shots=shots, separate_observables=True)
    pred = matcher.decode_batch(det)
    n_errors = int(np.sum(pred[:, 0] != obs[:, 0]))
    return n_errors / shots

def estimate_tstate(distance, rounds, p, n_shots, seed):
    """Return (E_T, PL) from a single experiment."""
    pl = run_memory_plus(distance, rounds, p, n_shots, seed)
    e_t = E_IDEAL * (1.0 - 2.0 * pl)
    return e_t, pl

def validate_variance(distance, p, n_shots, n_repeats, rounds=None):
    if rounds is None:
        rounds = 3 * distance

    e_vals = []
    pl_vals = []

    for r in range(n_repeats):
        seed = RNG_SEED + r * 1000
        e_t, pl = estimate_tstate(distance, rounds, p, n_shots, seed)
        e_vals.append(e_t)
        pl_vals.append(pl)

    e_vals = np.array(e_vals)
    pl_vals = np.array(pl_vals)

    emp_var = np.var(e_vals, ddof=1)
    mean_pl = np.mean(pl_vals)

    # Theoretical variance: Var(E_T) = 2 * PL * (1 - PL) / n_shots
    theo_var = 2.0 * mean_pl * (1.0 - mean_pl) / n_shots

    ratio = emp_var / theo_var if theo_var > 0 else float('inf')

    return {
        "distance": distance,
        "mean_E_T": np.mean(e_vals),
        "mean_PL": mean_pl,
        "empirical_var": emp_var,
        "theoretical_var": theo_var,
        "ratio": ratio,
    }

# Run the validation at three distances
for d in [13]:
    res = validate_variance(
        distance=d,
        p=0.0022,
        n_shots=1_000_000,
        n_repeats=30
    )
    print(res)